In [ ]:
# 1. 구글 드라이브 마운트 (학습 결과 저장용)
from google.colab import drive
drive.mount('/content/drive')

# 2. GPU 할당 확인
!nvidia-smi

# 3. Unsloth 및 관련 최신 라이브러리 설치 (T4 최적화)
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thu Sep 17 13:36:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|          

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048  # 뉴스 기사 + 요약문 보기에 충분한 길이
dtype = None  # None 설정 시 자동 감지 (T4는 Float16)
load_in_4bit = True  # QLoRA 메모리 절약

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# LoRA 어댑터 설정 (가중치 효율 학습)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # 8, 16, 32 중 선택 (16 추천)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # VRAM 대폭 절약
    random_state = 3407,
)

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1551: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.9.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
import json
from datasets import Dataset

# 1) dataset.jsonl 파일 불러오기 (미리 코랩에 업로드해 두세요)
data = []
with open('drive/MyDrive/Trender/trained_data/combined_dataset.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

# 2) 프롬프트 템플릿 정의 (학습 및 추론 시 동일한 입출력 구조 유지)
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # 문장 끝 알림 토큰

def format_prompts(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }

# Hugging Face Dataset 변환 및 포맷 적용
dataset = Dataset.from_list(data)
dataset = dataset.map(format_prompts, batched = True)

Map:   0%|          | 0/424 [00:00<?, ? examples/s]

In [ ]:
import torch
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        packing = False,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# 학습 시작
trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/424 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 424 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.778445
2,1.788965
3,1.807445
4,1.868036
5,1.741807
6,1.849905
7,1.662143
8,1.651469
9,1.606555
10,1.595795


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [ ]:
FastLanguageModel.for_inference(model) # 추론 속도 2배 향상 모드

sample_news = """네이버 사상 최대 실적…AI 쇼핑의 힘
입력2026.04.30 18:01 수정2026.05.01 00:44 지면A11
기사 스크랩
댓글
기사 공유
글자크기 조절
프린트
이영애
이영애
구글 검색 선호 출처로 추가
Google 검색에서 한국경제 기사를 더 자주 볼 수 있습니다.

1분기 매출 3조2411억

영업익 5418억…전년보다 7%↑
글로벌 중고 플랫폼 매출 급성장
스토어앱도 방문자 23% 늘어
"하반기 무제한 무료배송 도입"
지난해 검색에서 쇼핑으로 성장 축을 이동한다고 선언한 네이버가 본격적으로 성과를 내기 시작했다. 네이버플러스 스토어 앱이 1년 만에 소매시장에서 자리를 잡으며 올해 1분기 매출이 사상 최대를 기록한 것이다. 네이버는 쇼핑에 자체 인공지능(AI) 기술을 얹어 국내 최강자인 쿠팡을 추격한다는 계획이다.
◇자리 잡은 스토어 앱
네이버 사상 최대 실적…AI 쇼핑의 힘이미지 크게보기
네이버는 올 1분기 매출(연결기준)이 전년보다 16.3% 증가한 3조2411억원, 영업이익은 7.2% 늘어난 5418억원으로 집계됐다고 30일 발표했다. 매출은 사상 최고치이며, 1분기 기준 3조원을 넘은 것도 이번이 처음이다.

ADVERTISEMENT


매출이 증가한 이유는 커머스(쇼핑)에 있다. 약 1000만 명으로 알려진 네이버플러스 멤버십 가입자와 검색으로 유입된 이용자들이 네이버에서 물건을 많이 샀다는 의미다. 쇼핑이 포함된 네이버 플랫폼 서비스 1분기 매출은 4453억원으로 1년 전보다 36% 급증했다.

지난해 3월 출시된 네이버플러스 스토어 앱도 자리를 잡는 분위기다. 네이버 관계자는 “1분기 스토어 앱 이용자의 체류 시간은 출시 초기 대비 두 배 이상 늘었고, 재방문자는 전 분기 대비 23% 증가했다”며 “특히 실제 구입으로 이어지는 구매율이 웹보다 84% 높은 것으로 나와 앱이 단순 유입 채널을 넘어 실제 매출을 만드는 핵심 역할을 하기 시작했다”고 설명했다. 네이버는 스마트스토어나 스토어 앱 거래액은 공개하지 않는다. 쇼핑 사업의 선전으로 네이버페이(N페이) 결제액은 전년 동기 대비 23.4% 증가한 24조2000억원을 기록했다.

최수연 네이버 대표는 이날 실적 콘퍼런스콜에서 “핵심 상품의 배송 전환 지원과 물류 직계약 확대를 추진 중”이라며 “하반기에는 멤버십과 연계된 ‘무제한 무료 배송’을 도입해 이용자 록인(lock-in) 효과를 극대화할 방침”이라고 말했다.

ADVERTISEMENT


마진이 높은 쇼핑 사업 덕에 지난 1분기 영업이익률은 16.7%에 달했다. 네이버는 1분기 제25회 밀라노 동계올림픽과 e스포츠 중계권 확보 등에 수백억원을 썼다.
◇“쇼핑에 AI 기능 얹는다”
기존 핵심 사업이던 광고 사업은 부진했다. 온라인 광고 시장 둔화 속에 1분기 매출 1조3495억원을 냈다. 매출 증가율은 9.3%로 전체 매출 증가율(16.3%)에 크게 못 미쳤다.

다만 네이버는 AI를 기반으로 한 실질적인 성과가 조금씩 나오고 있다고 했다. 2월 선보인 쇼핑 AI 에이전트는 출시 대비 재방문자가 네 배 이상 늘었다. 최 대표는 “광고 매출 증가분 중 AI 기술 기여도가 50% 이상”이라고 강조했다. AI 기술과 광고 접목의 시너지를 확인한 네이버는 2분기부터 AI 수익 모델 고도화에 나서겠다는 계획이다.

ADVERTISEMENT


네이버는 광고주와 사업주를 위한 에이전트도 하반기에 출시해 생태계를 고도화할 계획이다. 특히 AI 기능을 쇼핑에도 적극 접목해 시장 지배력을 확대해나갈 방침이다.

최 대표는 “검색에서 발견·탐색을 거쳐 실제 구매와 예약까지 이어지는 끊김 없는 ‘실행형 AI’ 전략을 통해 소비 시장을 공략할 것”이라며 “사용자 만족도를 높여 수익화 확대로 이어지는 선순환 구조를 구축하고 글로벌 도전 영역에서도 지속적으로 기회를 발굴하겠다”고 말했다.

이날 네이버 주가는 4.09% 하락한 21만1000원에 마감했다. 견조한 실적에도 네이버의 추가 성장 스토리와 AI 전략에 대한 의구심, 비용 부담, 두나무와의 합병 지연 등이 부각된 데 따른 것으로 분석된다.

이영애 기자 0ae@hankyung.com"
"""

inputs = tokenizer(
[
    alpaca_prompt.format(
        "다음 뉴스 기사를 분석하여 핵심 요약, 감성(호재/악재/중립), 그리고 이유를 작성하세요.", # 학습할 때 사용한 Instruction으로 수정
        sample_news, # Input
        "", # Response (빈값)
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
다음 뉴스 기사를 분석하여 핵심 요약, 감성(호재/악재/중립), 그리고 이유를 작성하세요.

### Input:
네이버 사상 최대 실적…AI 쇼핑의 힘
입력2026.04.30 18:01 수정2026.05.01 00:44 지면A11
기사 스크랩
댓글
기사 공유
글자크기 조절
프린트
이영애 
이영애
구글 검색 선호 출처로 추가
Google 검색에서 한국경제 기사를 더 자주 볼 수 있습니다.

1분기 매출 3조2411억

영업익 5418억…전년보다 7%↑
글로벌 중고 플랫폼 매출 급성장
스토어앱도 방문자 23% 늘어
"하반기 무제한 무료배송 도입"
지난해 검색에서 쇼핑으로 성장 축을 이동한다고 선언한 네이버가 본격적으로 성과를 내기 시작했다. 네이버플러스 스토어 앱이 1년 만에 소매시장에서 자리를 잡으며 올해 1분기 매출이 사상 최대를 기록한 것이다. 네이버는 쇼핑에 자체 인공지능(AI) 기술을 얹어 국내 최강자인 쿠팡을 추격한다는 계획이다.
◇자리 잡은 스토어 앱
네이버 사상 최대 실적…AI 쇼핑의 힘이미지 크게보기
네이버는 올 1분기 매출(연결기준)이 전년보다 16.3% 증가한 3조2411억원, 영업이익은 7.2% 늘어난 5418억원으로 집계됐다고 30일 발표했다. 매출은 사상 최고치이며, 1분기 기준 3조원을 넘은 것도 이번이 처음이다.

ADVERTISEMENT


매출이 증가한 이유는 커머스(쇼핑)에 있다. 약 1000만 명으로 알려진 네이버플러스 멤버십 가입자와 검색으로 유입된 이용자들이 네이버에서 물건을 많이 샀다는 의미다. 쇼핑이 포함된 네이버 플랫폼 서비스 1분기 매출은 4453억원으로 1년 전보다 36% 급증했다.

지난해 3월 출시된 네

In [ ]:
import shutil
import os

# 1. 코랩 로컬 디스크 경로 설정 (기본 80~100GB 여유 공간 제공)
local_save_path = "/content/temp_gguf"
drive_save_path = "/content/drive/MyDrive/my_stock_sllm_gguf"

# 2. 로컬 디스크에 GGUF 변환 실행
model.save_pretrained_gguf(
    local_save_path,
    tokenizer,
    quantization_method = "q4_k_m",
)

# 3. 구글 드라이브 폴더 생성
os.makedirs(drive_save_path, exist_ok=True)

# 4. 최종 변환된 .gguf 파일만 구글 드라이브로 복사 (약 4~5GB 수준)
for file in os.listdir(local_save_path):
    if file.endswith(".gguf"):
        src = os.path.join(local_save_path, file)
        dst = os.path.join(drive_save_path, file)
        shutil.copy(src, dst)
        print(f"구글 드라이브 저장 완료: {dst}")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/temp_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

... (14 outputs left out)



Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [07:26<00:00, 111.74s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [05:44<00:00, 86.24s/it]


Unsloth: Merge process complete. Saved to `/content/temp_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10909-mix-bea84f7 (app-b10909-mix-bea84f7-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/temp_gguf_gguf/qwen2.5-7b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed suc

In [ ]:
import os
import shutil

# Unsloth가 실제로 파일을 생성한 경로 (_gguf가 붙음)
actual_save_path = "/content/temp_gguf_gguf"
drive_save_path = "/content/drive/MyDrive/my_stock_sllm_gguf"

# 구글 드라이브 폴더 생성
os.makedirs(drive_save_path, exist_ok=True)

# GGUF 모델 및 Ollama 서빙용 Modelfile 복사
for file in os.listdir(actual_save_path):
    if file.endswith(".gguf") or file == "Modelfile":
        src = os.path.join(actual_save_path, file)
        dst = os.path.join(drive_save_path, file)
        print(f"복사 중: {file} ...")
        shutil.copy(src, dst)
        print(f"구글 드라이브 저장 완료: {dst}")

print("\n모든 파일이 성공적으로 구글 드라이브에 저장되었습니다!")

복사 중: Modelfile ...
구글 드라이브 저장 완료: /content/drive/MyDrive/my_stock_sllm_gguf/Modelfile
복사 중: qwen2.5-7b-instruct.Q4_K_M.gguf ...
구글 드라이브 저장 완료: /content/drive/MyDrive/my_stock_sllm_gguf/qwen2.5-7b-instruct.Q4_K_M.gguf

모든 파일이 성공적으로 구글 드라이브에 저장되었습니다!


In [ ]:
!pip install --upgrade "gguf>=0.10.0"

In [ ]:
import json
import random
import torch
import packaging.version
from transformers import AutoModelForCausalLM, AutoTokenizer

# 🛠️ GGUF 메타데이터 버전('N/A') 파싱 에러 우회 패치
_original_version = packaging.version.Version

class SafeVersion(_original_version):
    def __init__(self, version):
        try:
            super().__init__(version)
        except packaging.version.InvalidVersion:
            super().__init__("0.0.0")

packaging.version.Version = SafeVersion

# 1) 경로 및 파일 설정
folder_path = "/content/drive/MyDrive/my_stock_sllm_gguf"
file_name = "qwen2.5-7b-instruct.Q4_K_M.gguf"

# 2) GGUF 모델 및 토크나이저 로드
model = AutoModelForCausalLM.from_pretrained(
    folder_path,
    gguf_file=file_name,
    torch_dtype="auto",
    device_map="cuda"
)

try:
    tokenizer = AutoTokenizer.from_pretrained(folder_path, gguf_file=file_name)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

# 3) dataset.jsonl 불러오기
data = []
with open('/content/drive/MyDrive/Trender/trained_data/combined_dataset.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
"""

def run_hf_gguf_evaluation(dataset_list, num_samples=3, random_seed=None):
    if random_seed is not None:
        random.seed(random_seed)

    k = min(num_samples, len(dataset_list))
    sampled_data = random.sample(dataset_list, k)

    for idx, item in enumerate(sampled_data, 1):
        instruction = item.get("instruction", "")
        input_text = item.get("input", "")
        ground_truth = item.get("output", "")

        prompt = alpaca_prompt.format(instruction, input_text)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=512)

        decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

        if "### Response:" in decoded_output:
            predicted_response = decoded_output.split("### Response:")[-1].strip()
        else:
            predicted_response = decoded_output.strip()

        print(f"\n==================== [ 샘플 {idx}/{k} GGUF 평가 ] ====================")
        print(f"📌 [Instruction]:\n{instruction}")
        print(f"\n📌 [Input (미리보기)]:\n{input_text[:150]}..." if len(input_text) > 150 else f"\n📌 [Input]:\n{input_text}")
        print(f"\n🎯 [정답 (Ground Truth)]:\n{ground_truth}")
        print(f"\n🤖 [모델 생성 (Prediction)]:\n{predicted_response}")
        print("=" * 65)

# 무작위 3개 샘플 실행
run_hf_gguf_evaluation(data, num_samples=3, random_seed=42)

Converting and de-quantizing GGUF tensors...:   0%|          | 0/339 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [6]:
# 1. zstd 패키지 설치 및 Ollama 재설치
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install pyngrok

# 2. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Fetched 1,578 B in 1s (1,870 B/s)
Reading package lists... Done
W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of c

In [1]:
import os
import subprocess
import time

# 기존 시스템 PATH 유지하며 환경변수 추가
env = os.environ.copy()
env["OLLAMA_FLASH_ATTENTION"] = "1"
env["OLLAMA_KEEP_ALIVE"] = "24h"

# Ollama 서버 백그라운드 실행
subprocess.Popen(["ollama", "serve"], env=env)
time.sleep(5) # 서버 가동 대기

# 정상 실행 여부 확인
!ollama --version

ollama version is 0.34.2


In [26]:
# Modelfile 작성 (드라이브 내 gguf 파일 경로 지정)
modelfile_content = """
FROM /content/drive/MyDrive/my_stock_sllm_gguf/qwen2.5-7b-instruct.Q4_K_M.gguf
PARAMETER num_gpu 99
PARAMETER num_ctx 1024
PARAMETER num_predict 256
"""

with open("Modelfile", "w") as f:
    f.write(modelfile_content)

# Ollama 모델 생성
!ollama create stock-llm -f ./Modelfile

In [27]:
from pyngrok import ngrok

# 1. ngrok 대시보드 토큰 등록
NGROK_TOKEN = "3JPPy6G9Awcu0OMi04uJyb54Gtu_c6kGyzZx1Tb8yjhVUcy"  # 본인 Authtoken 입력
ngrok.set_auth_token(NGROK_TOKEN)

# 2. 기존 연결 종료 후 host_header 옵션 적용하여 11434 포트 연결
ngrok.kill()
public_url = ngrok.connect(11434, "http", host_header="localhost:11434")

print(f"\n🚀 최적화 적용된 Ollama 서버 URL: {public_url.public_url}")


🚀 최적화 적용된 Ollama 서버 URL: https://buddy-proofread-jingling.ngrok-free.dev


finetuning 안된 기본 qwen2.5-7b 모델로 서버 열기

In [7]:
modelfile_content = """
FROM qwen2.5:7b
PARAMETER num_gpu 99
PARAMETER num_ctx 1024
PARAMETER num_predict 256
"""

with open("Modelfile", "w") as f:
    f.write(modelfile_content)

# 모델 이름은 소문자 및 하이픈(-)만 사용
!ollama create qwen2.5-base -f ./Modelfile

In [8]:
from pyngrok import ngrok

# ngrok 대시보드(https://dashboard.ngrok.com)에서 발급받은 실제 Authtoken 입력
NGROK_TOKEN = "3JPPy6G9Awcu0OMi04uJyb54Gtu_c6kGyzZx1Tb8yjhVUcy"
ngrok.set_auth_token(NGROK_TOKEN)

# 기존 프로세스 정리 후 host-header 우회 설정 연결
ngrok.kill()
public_url = ngrok.connect(11434, "http", host_header="localhost:11434")

print(f"\n🚀 최종 Ollama 서버 엔드포인트 URL: {public_url.public_url}")


🚀 최종 Ollama 서버 엔드포인트 URL: https://buddy-proofread-jingling.ngrok-free.dev
